# Markov chain Attribution model

Idea: Calculate the transition probabilities between each state (see a TV/PROGRAMMATIC + Add to cart or visit the site + outcome: Order or Lost customer)

Then: Calculate the effect of removing one of the canals on the global conversion rate to determine the importance of each step in the consumer journey

### I. Importing data + grouping by customer_id and chronological order

In [2]:
import pandas as pd
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile('data/customer_journey.parquet')

aggregated_data = []
cols = [ # We only care here about event tye, not financial result
    "customer_id",
    "timestamp_utc",
    "channel",
    "event_type"
]

for batch in parquet_file.iter_batches(batch_size=500_000,columns=cols):
   
    chunk = batch.to_pandas()
    
    aggregated_data.append(chunk)

# Fusion finale
df = pd.concat(aggregated_data, ignore_index=True)

In [3]:
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])

df = df.sort_values(
    by=["customer_id", "timestamp_utc"]
)

### II. Defining each state and computing each consumer's journey

In [4]:
import numpy as np

media_channels = ["TV", "PROGRAMMATIC"]
retail_events = ["Product Page View", "Add to cart"]

df_model = df[
    (df["channel"].isin(media_channels)) |
    (df["event_type"].isin(retail_events + ['Order']))
].copy()

df_model["state"] = np.where(
    df_model["event_type"] == "Order",
    "Order",
    np.where(
        df_model["channel"].isin(media_channels),
        df_model["channel"],
        df_model["event_type"]
    )
)

In [5]:
paths = (
    df_model
    .groupby("customer_id")["state"]
    .apply(list)
)

In [6]:
def build_paths_multiple_orders(states):
    """
    Converts a list of states into multiple paths,
    one per order, truncating at each Order.
    """
    paths = []
    current_path = ["Start"]
    
    for state in states:
        current_path.append(state)
        if state == "Order":
            paths.append(current_path)
            current_path = ["Start"]  # new journey for next order
    
    # if no orders at all, append Null
    if not paths:
        paths.append(current_path + ["Null"])
    
    return paths

# Apply to your grouped sequences
all_paths = []

for path in paths:  # `paths` from groupby(customer_id)
    subpaths = build_paths_multiple_orders(path)
    all_paths.extend(subpaths)


### III. Computing the Markov transition matrix 

In [7]:
from collections import Counter

transition_counter = Counter()

for path in all_paths:
    for i in range(len(path) - 1):
        transition_counter[(path[i], path[i+1])] += 1

transitions_df = (
    pd.DataFrame(
        ((k[0], k[1], v) for k, v in transition_counter.items()),
        columns=["from", "to", "count"]
    )
)

In [8]:
transitions_df

,from,to,count
0,Start,TV,1650560
1,TV,TV,3043722
2,TV,Null,1558080
3,Start,PROGRAMMATIC,4260968
4,PROGRAMMATIC,Null,4354528
5,Start,Product Page View,1165170
6,Product Page View,Product Page View,3037046
7,Product Page View,Null,506600
8,PROGRAMMATIC,PROGRAMMATIC,9531914
9,Start,Add to cart,555212


In [9]:
transition_matrix = (
    transitions_df
    .pivot(index="from", columns="to", values="count")
    .fillna(0)
)

transition_matrix = transition_matrix.div(
    transition_matrix.sum(axis=1), axis=0
)

In [10]:
transition_matrix

to,Add to cart,Null,Order,PROGRAMMATIC,Product Page View,TV
from,,,,,,
Add to cart,0.154463,0.016510,0.350146,0.031972,0.442461,0.004448
PROGRAMMATIC,0.005369,0.298423,0.003379,0.653237,0.029505,0.010088
Product Page View,0.146898,0.093550,0.098093,0.093483,0.560828,0.007148
Start,0.070654,0.000000,0.028801,0.542230,0.148274,0.210042
TV,0.002442,0.318771,0.001499,0.048737,0.005830,0.622721


In [11]:
states = transition_matrix.index.tolist()

absorbing_states = ["Order", "Null"]
transient_states = [s for s in states if s not in absorbing_states]


In [12]:
all_states = transient_states + absorbing_states
transition_matrix = transition_matrix.reindex(index=all_states, columns=all_states, fill_value=0)

In [13]:
transition_matrix

to,Add to cart,PROGRAMMATIC,Product Page View,Start,TV,Order,Null
from,,,,,,,
Add to cart,0.154463,0.031972,0.442461,0.0,0.004448,0.350146,0.016510
PROGRAMMATIC,0.005369,0.653237,0.029505,0.0,0.010088,0.003379,0.298423
Product Page View,0.146898,0.093483,0.560828,0.0,0.007148,0.098093,0.093550
Start,0.070654,0.542230,0.148274,0.0,0.210042,0.028801,0.000000
TV,0.002442,0.048737,0.005830,0.0,0.622721,0.001499,0.318771
Order,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
Null,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000


### IV: Compunting global conversion probabilities

In [14]:
Q = transition_matrix.loc[transient_states, transient_states]
R = transition_matrix.loc[transient_states, absorbing_states]

I = np.eye(len(Q))
N = np.linalg.inv(I - Q)

In [15]:
start_idx = transient_states.index("Start")
order_idx = absorbing_states.index("Order")

conversion_prob = (N @ R).iloc[start_idx, order_idx]
conversion_prob

np.float64(0.1795426757570914)

### V. Computing the final effect of each marketing channel 

In [16]:
def removal_effect(channel, transition_matrix):
    tm = transition_matrix.copy()

    if channel not in tm.index:
        return 0

    tm = tm.drop(index=channel, columns=channel, errors="ignore")

    tm = tm.div(tm.sum(axis=1), axis=0)

    states = tm.index.tolist()
    absorbing = ["Order", "Null"]
    transient = [s for s in states if s not in absorbing]

    Q = tm.loc[transient, transient]
    R = tm.loc[transient, absorbing]

    N = np.linalg.inv(np.eye(len(Q)) - Q)

    start_idx = transient.index("Start")
    order_idx = absorbing.index("Order")

    return (N @ R).iloc[start_idx, order_idx]


In [17]:
base_cvr = conversion_prob

effects = {}

for channel in ["TV", "PROGRAMMATIC", "Product Page View", "Add to cart"]:
    cvr_removed = removal_effect(channel, transition_matrix)
    effects[channel] = base_cvr - cvr_removed

effects

{'TV': np.float64(-0.045958172942340336),
 'PROGRAMMATIC': np.float64(-0.2072340748146486),
 'Product Page View': np.float64(0.053814666907398084),
 'Add to cart': np.float64(0.06593884671283701)}

In [18]:
total_effect = sum(effects.values())

attribution = {
    k: v / total_effect
    for k, v in effects.items()
}

full_contribution = pd.DataFrame.from_dict(
    attribution, orient="index", columns=["Contribution"]
).sort_values("Contribution", ascending=False)

full_contribution

,Contribution
PROGRAMMATIC,1.553028
TV,0.344414
Product Page View,-0.403291
Add to cart,-0.494151


### VI. Plotting the results

In [19]:
# Attribution table for media channels
media_attribution = pd.DataFrame({
    "Channel": ["PROGRAMMATIC", "TV"],
    "Contribution": [1.553028, 0.344414]
})

# Optional: full contribution table including funnel steps
full_contribution = pd.DataFrame({
    "State": ["PROGRAMMATIC", "TV", "Product Page View", "Add to cart"],
    "Contribution": [1.553028, 0.344414, -0.403291, -0.494151]
})

In [20]:
import plotly.express as px
import plotly.graph_objects as go

fig = px.bar(
    media_attribution,
    x="Channel",
    y="Contribution",
    text="Contribution",
    color="Channel",
    color_discrete_sequence=px.colors.qualitative.Vivid
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(
    title="Marketing Attribution by Channel (Incremental Contribution)",
    yaxis_title="Contribution",
    xaxis_title="Channel",
    uniformtext_minsize=8
)

fig.show()


In [21]:
fig = px.bar(
    full_contribution,
    x="State",
    y="Contribution",
    color="Contribution",
    color_continuous_scale=px.colors.diverging.RdBu,
    text="Contribution"
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(
    title="Contribution of Funnel Steps (Positive = Media Credit, Negative = Funnel Effect)",
    yaxis_title="Contribution",
    xaxis_title="State",
    coloraxis_colorbar=dict(title="Contribution")
)

fig.show()

In [27]:
import plotly.graph_objects as go
from collections import Counter

# Flatten paths into transitions with counts
transitions_counter = Counter()
for path in all_paths:
    for i in range(len(path)-1):
        transitions_counter[(path[i], path[i+1])] += 1

# Prepare Sankey inputs
all_states = list({s for path in all_paths for s in path})
state_indices = {state: i for i, state in enumerate(all_states)}

sources = [state_indices[src] for src, tgt in transitions_counter.keys()]
targets = [state_indices[tgt] for src, tgt in transitions_counter.keys()]
values = list(transitions_counter.values())

fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        label=all_states,
        pad=15,
        thickness=20,
        color="lightblue"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig_sankey.update_layout(
    title_text="User Journey Funnel Sankey Diagram (Channels → Funnel → Conversion)",
    font_size=12
)

fig_sankey.show()


In [24]:
# Assume you have a dataframe: paths_df = ["path_length", "conversions"]
paths_df = pd.DataFrame({
    "Path Length": [1, 2, 3, 4, 5],
    "Conversions": [2000, 4500, 8000, 12000, 14000]
})

fig = px.line(
    paths_df,
    x="Path Length",
    y="Conversions",
    markers=True,
    title="Cumulative Conversions by Number of Touchpoints"
)
fig.update_layout(xaxis_title="Number of Touchpoints", yaxis_title="Conversions")
fig.show()


In [25]:
fig = px.imshow(
    transition_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='Viridis',
    title="Transition Probability Matrix (Markov)"
)
fig.update_layout(xaxis_title="To State", yaxis_title="From State")
fig.show()


In [32]:
import pandas as pd
import plotly.graph_objects as go

# Your matrix
tm = transition_matrix  # use your DataFrame

# Flatten into source, target, value
sources = []
targets = []
values = []

for from_state in tm.index:
    for to_state in tm.columns:
        prob = tm.loc[from_state, to_state]
        if prob > 0:
            sources.append(from_state)
            targets.append(to_state)
            values.append(prob)

# Map state names to indices
all_states = list(tm.index)
state_to_idx = {state: i for i, state in enumerate(all_states)}

source_idx = [state_to_idx[s] for s in sources]
target_idx = [state_to_idx[t] for t in targets]

# Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_states,
        color="lightblue"
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=['rgba(0,0,255,{:.2f})'.format(v) for v in values],  # optional: alpha by probability
        hovertemplate=[f"{s} → {t}: {v:.2f}" for s,t,v in zip(sources, targets, values)]
    )
)])

fig.update_layout(
    title_text="Markov Transition Probabilities (Sankey-style)",
    font_size=12,
    width=1000,
    height=600
)

fig.show()


In [28]:
import networkx as nx

G = nx.DiGraph()

# Add edges with transition probability as weight
for from_state in transition_matrix.index:
    for to_state in transition_matrix.columns:
        prob = transition_matrix.loc[from_state, to_state]
        if prob > 0:
            G.add_edge(from_state, to_state, weight=prob)

In [29]:
pos = nx.spring_layout(G, seed=42)  # deterministic layout

edge_x = []
edge_y = []
edge_width = []

for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    # scale width by probability
    edge_width.append(edge[2]['weight'] * 10)

node_x = []
node_y = []
for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

node_text = [f"{node}" for node in G.nodes()]

In [31]:
fig = go.Figure()

# Add edges
for i, edge in enumerate(G.edges(data=True)):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    fig.add_trace(go.Scatter(
        x=[x0, x1],
        y=[y0, y1],
        line=dict(width=edge[2]['weight']*10, color='blue'),
        hoverinfo='text',
        text=f"{edge[0]} → {edge[1]}: {edge[2]['weight']:.2f}",
        mode='lines'
    ))

# Add nodes
fig.add_trace(go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=node_text,
    textposition="top center",
    hoverinfo='text',
    marker=dict(size=40, color='lightgreen')
))

fig.update_layout(
    title="Markov Transition Probabilities Between States",
    showlegend=False,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    width=900,
    height=700
)

fig.show()
